### **Autor:** David Roca Tauste

---
---
# **ACTIVIDAD 3: DETECTANDO SPAM EN MENSAJES SMS**
---
---

## CARGAR Y EXPLORAR LOS DATOS

In [1]:
import pandas as pd

sms_spam = pd.read_csv(
    "./SMSSpamCollection.csv", sep="\t", header=None, names=["Label", "SMS"]
)
print("== Dimensiones: ", sms_spam.shape)
print("== Primeros 5 ejemplos:\n", sms_spam.head())
print("== Información de las columnas:")
print(sms_spam.info())

== Dimensiones:  (5572, 2)
== Primeros 5 ejemplos:
   Label                                                SMS
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...
== Información de las columnas:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Label   5572 non-null   object
 1   SMS     5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB
None


In [2]:
print("== Porcentajes de spam y ham:")
print( sms_spam['Label'].value_counts(normalize=True) )

== Porcentajes de spam y ham:
Label
ham     0.865937
spam    0.134063
Name: proportion, dtype: float64


## PREPARAR LOS DATOS DE ENTRENAMIENTO Y TEST

In [3]:
# Dividir en train + test
datos = sms_spam.sample(frac=1, random_state=1)  # Aleatorizar dataset
indices = round(len(datos) * 0.8)  # Calcula índices división
train = datos[:indices].reset_index(drop=True)
test = datos[indices:].reset_index(drop=True)
print("== Dimensiones de train:", train.shape)
print("== Dimensiones de test:", test.shape)
print("== Porcentajes de spam en datos train:")
print(train["Label"].value_counts(normalize=True))
print("== Porcentajes de spam en datos test:")
print(test["Label"].value_counts(normalize=True))

== Dimensiones de train: (4458, 2)
== Dimensiones de test: (1114, 2)
== Porcentajes de spam en datos train:
Label
ham     0.86541
spam    0.13459
Name: proportion, dtype: float64
== Porcentajes de spam en datos test:
Label
ham     0.868043
spam    0.131957
Name: proportion, dtype: float64


## LIMPIEZA DE DATOS

In [4]:
train_antes_limpieza = train.copy()

In [5]:
# Limpieza de datos
train["SMS"] = train["SMS"].str.replace("\W", " ")  # Elimina signos puntuación
train["SMS"] = train["SMS"].str.lower()  # Convierte a minúsculas

<>:2: SyntaxWarning: invalid escape sequence '\W'
<>:2: SyntaxWarning: invalid escape sequence '\W'
C:\Users\davil\AppData\Local\Temp\ipykernel_5004\2292733033.py:2: SyntaxWarning: invalid escape sequence '\W'
  train["SMS"] = train["SMS"].str.replace("\W", " ")  # Elimina signos puntuación


## ENTREGA 8: Modifica este trozo de código y añade dos sentencias para que muestre las dos primeras filas de train antes y después de aplicar la limpieza de datos como se ve en la figura de abajo para comprobar que efectivamente eliminas los signos de puntuación y conviertes a minúsculas. Si no consigue hacerlo intenta algunas de estas modificaciones:
- Importa re (expresiones regulares) y string y sustituye la línea 27 por esta: train['SMS'] = re.sub('[%s]' % re.escape(string.punctuation), ' ', train['SMS'].str)
- Importa string y sustituye la línea 27 por esta: train['SMS'] = train['SMS'].str.replace('[{}]'.format(string.punctuation), ' ', regex=True)
- A la función replace() de la línea 27 le añades el parámetro: regex=True

In [6]:
print("2 primeras filas antes de la limpieza:")
train_antes_limpieza.head(2)

2 primeras filas antes de la limpieza:


,Label,SMS
0,ham,"Yep, by the pretty sculpture"
1,ham,"Yes, princess. Are you going to make me moan?"


In [7]:
import string

# También funciona, pero muestra avisos, por eso utilizo la otra opción
#train["SMS"] = train["SMS"].str.replace("\W", " ", regex=True)
train['SMS'] = train['SMS'].str.replace('[{}]'.format(string.punctuation), ' ', regex=True)

print("2 primeras filas después de la limpieza:")
train.head(2)

2 primeras filas después de la limpieza:


,Label,SMS
0,ham,yep by the pretty sculpture
1,ham,yes princess are you going to make me moan


---
## CREAR EL VOCABULARIO Y TRANSFORMAR LOS DATOS

In [8]:
# Crear el vocabulario
train["SMS"] = train["SMS"].str.split()
vocabulario = []
for sms in train["SMS"]:
    for palabra in sms:
        vocabulario.append(palabra)
vocabulario = list(set(vocabulario))
print(f"Hay {len(vocabulario)} palabras distintas en los mensajes de train.")

Hay 7860 palabras distintas en los mensajes de train.


In [9]:
palabra_contadores_por_sms = [{palabra: 0 for palabra in vocabulario} for _ in range(len(train['SMS']))]

for idx, sms in enumerate(train['SMS']):
    for p in sms:
        palabra_contadores_por_sms[idx][p] += 1

palabras = pd.DataFrame(palabra_contadores_por_sms)
print(palabras.head())

   lipo  infront  sophas  computational  air  7876150ppm  definitly  sky  \
0     0        0       0              0    0           0          0    0   
1     0        0       0              0    0           0          0    0   
2     0        0       0              0    0           0          0    0   
3     0        0       0              0    0           0          0    0   
4     0        0       0              0    0           0          0    0   

   oyster  hours  ...  enters  £50award  particularly  smartcall  sucks  \
0       0      0  ...       0         0             0          0      0   
1       0      0  ...       0         0             0          0      0   
2       0      0  ...       0         0             0          0      0   
3       0      0  ...       0         0             0          0      0   
4       0      0  ...       0         0             0          0      0   

   houseful  trained  diet  beneficiary  114  
0         0        0     0            0    0 

In [10]:
train = pd.concat([train, palabras], axis=1)
print(train.head())

  Label                                                SMS  lipo  infront  \
0   ham                  [yep, by, the, pretty, sculpture]     0        0   
1   ham  [yes, princess, are, you, going, to, make, me,...     0        0   
2   ham                    [welp, apparently, he, retired]     0        0   
3   ham                                           [havent]     0        0   
4   ham  [i, forgot, 2, ask, ü, all, smth, there, s, a,...     0        0   

   sophas  computational  air  7876150ppm  definitly  sky  ...  enters  \
0       0              0    0           0          0    0  ...       0   
1       0              0    0           0          0    0  ...       0   
2       0              0    0           0          0    0  ...       0   
3       0              0    0           0          0    0  ...       0   
4       0              0    0           0          0    0  ...       0   

   £50award  particularly  smartcall  sucks  houseful  trained  diet  \
0         0         

## CALCULAR VALORES DE LA FÓRMULA

In [11]:
# Calcular el modelo
sms_spam = train[train['Label'] == 'spam']
sms_ham = train[train['Label'] == 'ham']

p_spam = len(sms_spam) / len(train)
p_ham = len(sms_ham) / len(train)

n_spam = sms_spam['SMS'].apply(len).sum()
n_ham = sms_ham['SMS'].apply(len).sum()
n_vocabulary = len(vocabulario)
alfa = 1

In [12]:
# Inicializar y calcular los parámetros
param_spam = {palabra: 0 for palabra in vocabulario}
param_ham = {palabra: 0 for palabra in vocabulario}

for palabra in vocabulario:
    n_wi_spam = sms_spam[palabra].sum()
    p_wi_spam = (n_wi_spam + alfa) / (n_spam + alfa * n_vocabulary)
    param_spam[palabra] = p_wi_spam

    n_wi_ham = sms_ham[palabra].sum()
    p_wi_ham = (n_wi_ham + alfa) / (n_ham + alfa * n_vocabulary)
    param_ham[palabra] = p_wi_ham

## CLASIFICAR MENSAJES

In [15]:
import re

def clasifica(mensaje):
    mensaje = re.sub('\W', ' ', mensaje)
    mensaje = mensaje.lower().split()

    p_spam_mensaje = p_spam
    p_ham_mensaje = p_ham

    for palabra in mensaje:
        if palabra in param_spam:
            p_spam_mensaje *= param_spam[palabra]

        if palabra in param_ham:
            p_ham_mensaje *= param_ham[palabra]

    print('P(Spam|mensaje):', p_spam_mensaje)
    print('P(Ham|mensaje):', p_ham_mensaje)

    if p_ham_mensaje > p_spam_mensaje:
        print('Label: Ham')
    elif p_ham_mensaje < p_spam_mensaje:
        print('Label: Spam')
    else:
        print('Igual de probable, un humano debe decidir!')

<>:4: SyntaxWarning: invalid escape sequence '\W'
<>:4: SyntaxWarning: invalid escape sequence '\W'
C:\Users\davil\AppData\Local\Temp\ipykernel_5004\3109586246.py:4: SyntaxWarning: invalid escape sequence '\W'
  mensaje = re.sub('\W', ' ', mensaje)


In [16]:
clasifica('WINNER!! This is the secret code to unlock the money: C3421.')

P(Spam|mensaje): 1.3183946420838108e-25
P(Ham|mensaje): 1.9319859467201923e-27
Label: Spam


In [17]:
clasifica("Sounds good, Tom, then see u there")

P(Spam|mensaje): 2.395324109936206e-25
P(Ham|mensaje): 3.6803924043364484e-21
Label: Ham


## MEDIR LA EFICIENCIA CON ACCURACY

## ENTREGA 9: Completa la función clasifica_test() y usándola con los mensajes de test calcula el accuracy. Recuerda que accuracy = mensajes bien clasificados / total de mensajes.

In [19]:
def clasifica_test(mensaje):
    mensaje = re.sub("\W", " ", mensaje)
    mensaje = mensaje.lower().split()

    p_spam_mensaje = p_spam
    p_ham_mensaje = p_ham

    for palabra in mensaje:
        if palabra in param_spam:
            p_spam_mensaje *= param_spam[palabra]

        if palabra in param_ham:
            p_ham_mensaje *= param_ham[palabra]

    if p_ham_mensaje > p_spam_mensaje:
        return 'ham'
    elif p_ham_mensaje < p_spam_mensaje:
        return 'spam'
    else:
        return 'Igual de probable'

<>:2: SyntaxWarning: invalid escape sequence '\W'
<>:2: SyntaxWarning: invalid escape sequence '\W'
C:\Users\davil\AppData\Local\Temp\ipykernel_5004\1119121561.py:2: SyntaxWarning: invalid escape sequence '\W'
  mensaje = re.sub("\W", " ", mensaje)


### Usa estas sentencias y pasa captura del resultado:

In [20]:
test['prediccion'] = test['SMS'].apply(clasifica_test)
print("== Test con predicciones realizadas:\n", test.head())

== Test con predicciones realizadas:
   Label                                                SMS prediccion
0   ham          Later i guess. I needa do mcat study too.        ham
1   ham             But i haf enuff space got like 4 mb...        ham
2  spam  Had your mobile 10 mths? Update to latest Oran...       spam
3   ham  All sounds good. Fingers . Makes it difficult ...        ham
4   ham  All done, all handed in. Don't know if mega sh...        ham


### Calcula accuracy e indica cuál es con estas sentencias:

In [21]:
correctas = 0
total = test.shape[0]
for indice, fila in test.iterrows():
    if fila['Label'] == fila['prediccion']:
        correctas += 1

print(f"== Correctas {correctas} de {total} Accuracy: {correctas/total:.4f}")

== Correctas 1100 de 1114 Accuracy: 0.9874


El accuracy es: 0.9874